# Mini-Project 2: Uganda tax engine

This notebook demonstrates the typed, extensible tax classes in `src/tax.py`. It is an educational model, not tax advice. The resident PAYE schedule follows the assignment baseline and the non-resident schedule is taken from URA's FY2026/27 Employment Income guidance. Other baseline rates follow the assignment, with sources and assumptions recorded in the module and README. Confirm current tax-year rules before any real use.

In [ ]:
from datetime import date
from decimal import Decimal

from src.tax import (
    CorporateIncomeTax,
    Invoice,
    Money,
    PAYE,
    Payroll,
    RentalIncomeTax,
    ResidencyStatus,
    SupplyType,
    TaxContext,
    VAT,
    WithholdingTax,
)

def UGX(value: int | str) -> Money:
    return Money(Decimal(value))

AS_OF = date(2026, 10, 24)

## PAYE and payroll

The order is employee and employer NSSF calculation, optional Local Service Tax (LST) before PAYE, then net pay. The supplied baseline does not say employee NSSF is deductible for PAYE, so this model does not subtract it from PAYE taxable income. LST is configurable as an amount because the assignment gives no LST rate.

In [ ]:
payroll_input = TaxContext(
    gross_income=UGX(500_000),
    allowances=UGX(25_000),
    local_service_tax=UGX(5_000),
    residency=ResidencyStatus.RESIDENT,
    effective_date=AS_OF,
)
payroll = Payroll().calculate(payroll_input)
payroll

## VAT and corporate income tax

Invoice amounts are before VAT. Standard-rated sales create output VAT; standard-rated purchases create recoverable input VAT. Zero-rated and exempt invoices have no output VAT in this simplified model. Corporate chargeable income is revenue minus allowable deductions, floored at zero before applying the 30% baseline rate.

In [ ]:
vat_result = VAT().calculate(
    TaxContext(
        invoices=(
            Invoice(UGX(1_000_000), SupplyType.STANDARD_RATED, is_sale=True),
            Invoice(UGX(300_000), SupplyType.STANDARD_RATED, is_sale=False),
            Invoice(UGX(200_000), SupplyType.ZERO_RATED, is_sale=True),
            Invoice(UGX(100_000), SupplyType.EXEMPT, is_sale=True),
        )
    )
)
corporate_result = CorporateIncomeTax().calculate(
    TaxContext(revenue=UGX(10_000_000), allowable_deductions=UGX(3_000_000))
)
print("VAT:", vat_result)
print("Corporate income tax:", corporate_result)

## Extension A: withholding and rental income tax

The optional extension adds a configurable withholding tax calculator and an individual rental tax calculator. The sample rental threshold and rates follow the dated secondary reference listed in the module; verify them with URA for the applicable year.

In [ ]:
withholding = WithholdingTax().calculate(TaxContext(gross_income=UGX(1_000_000)))
rental = RentalIncomeTax().calculate(TaxContext(gross_income=UGX(4_000_000)))
print("Withholding tax:", withholding.amount)
print("Rental income tax:", rental.amount)

## Money, validation, and testing

Binary floating point cannot exactly represent many decimal fractions. For example, in Python `0.1 + 0.2 == 0.3` is false because the binary approximations add to a value slightly different from the binary approximation of `0.3`. In tax code, repeated fractional errors can accumulate or move an amount across a band edge. `Decimal("0.1") + Decimal("0.2") == Decimal("0.3")` is exact for these decimal inputs. `Money` rejects floats, rounds to whole UGX with `ROUND_HALF_UP`, and implements addition, multiplication, ordering, and a readable representation.

All calculators implement the abstract `Tax.calculate(ctx: TaxContext) -> TaxResult` contract. Invalid negative incomes raise `InvalidIncomeError`; unsupported PAYE dates raise `NoScheduleForDateError`; both inherit from `TaxError`. The automated suite includes every PAYE boundary requested in the assignment and Hypothesis checks that PAYE remains non-negative, monotonic with income, and no greater than income. Run `python -m pytest` from the repository root. The resident schedule is the provided assignment rule set (effective date labeled 1 July 2026 for this sample). For the schedule used here, the source date/version is recorded in `src/tax.py`; rates may change by tax year.

Inheritance is justified for the tax calculators because PAYE, NSSF, VAT, corporate income tax, and the extension classes share the `Tax.calculate` contract and can be substituted through one common interface. Payroll composes PAYE and NSSF rather than inheriting from either, because payroll coordinates separate calculations instead of being a kind of tax. `Money`, `Invoice`, and `TaxContext` are value/data objects; adding subclasses for each currency, invoice type, or employee status would add complexity without shared behavior, so the implementation deliberately uses composition and enums there.

Sources: assignment baseline in `Assignment_2_Advent_2026_MiniProjects.pdf`, page 4; [URA Employment Income guidance](https://ura.go.ug/en/employment-income/) (FY2026/27 PAYE schedule, accessed 24 October 2026); [PwC Uganda individual tax summary](https://taxsummaries.pwc.com/uganda/individual/taxes-on-personal-income) (rental and withholding examples, accessed 24 October 2026). The assignment baseline and other tax-year rules should be reviewed with URA before real use.